In [2]:
!pip install openai pandas


In [18]:
from openai import OpenAI
import pandas as pd

client = OpenAI(api_key="PUT YOUR API KEY HERE")


In [4]:
!unzip -o spider.zip -d spider


Archive:  spider.zip
  inflating: spider/spider/README.txt  
  inflating: spider/spider/database/academic/academic.sqlite  
  inflating: spider/spider/database/academic/schema.sql  
  inflating: spider/spider/database/activity_1/activity_1.sqlite  
  inflating: spider/spider/database/activity_1/schema.sql  
  inflating: spider/spider/database/aircraft/aircraft.sqlite  
  inflating: spider/spider/database/aircraft/schema.sql  
  inflating: spider/spider/database/allergy_1/allergy_1.sqlite  
  inflating: spider/spider/database/allergy_1/schema.sql  
  inflating: spider/spider/database/apartment_rentals/apartment_rentals.sqlite  
  inflating: spider/spider/database/apartment_rentals/schema.sql  
  inflating: spider/spider/database/architecture/architecture.sqlite  
  inflating: spider/spider/database/architecture/schema.sql  
  inflating: spider/spider/database/assets_maintenance/assets_maintenance.sqlite  
  inflating: spider/spider/database/assets_maintenance/schema.sql  
  inflating: s

In [16]:
import json

with open("spider/spider/dev.json") as f:
    spider_dev = json.load(f)

examples = []
for item in spider_dev[:10]: #Remove regment to eval on entire val data
    examples.append({
        "question": item["question"],
        "expected_sql": item["query"],
        "db_id": item["db_id"]
    })


In [6]:
import sqlite3

def get_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    schema_text = ""
    for (table_name,) in tables:
        schema_text += f"Table: {table_name}\n"
        cursor.execute(f"PRAGMA table_info({table_name});")
        for col in cursor.fetchall():
            col_name, col_type = col[1], col[2]
            schema_text += f"- {col_name} ({col_type})\n"
        schema_text += "\n"

    conn.close()
    return schema_text.strip()


In [12]:
# Set which model you want to use at model=XYZ:
# Options:
# - "gpt-4o"
# - "ft:gpt-3.5-turbo-0125:personal::BRP9mbO8"

def generate_sql(question, schema_text):
    prompt = f"""
You are an expert at converting questions into SQL queries.

Follow these strict rules:
- Do NOT add aliases (no AS ...) unless there are multiple tables, then alias as T1, T2, ...
- Use exact column and table names as in the schema
- Leave all column names as lowercase
- Match the expected SQL structure exactly (no extra formatting or comments)
- Only return raw SQL — no explanations, no Markdown formatting.

Schema:
{schema_text}

Translate this question into SQL:
"{question}"

SQL:
"""
    response = client.chat.completions.create(
        model="ft:gpt-3.5-turbo-0125:personal::BRP9mbO8",
        messages=[
            {"role": "system", "content": "You are a precise SQL translator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
    )
    output = response.choices[0].message.content.strip()

    # Clean markdown fences
    if output.startswith("```sql"):
        output = output.replace("```sql", "").replace("```", "").strip()
    elif output.startswith("```"):
        output = output.replace("```", "").strip()

    return output


In [13]:
import re

def normalize_sql(sql):
    sql = sql.strip().lower()                        # Case-insensitive
    sql = re.sub(r'\s+', ' ', sql)                   # Collapse all whitespace to single space
    sql = sql.replace(" ;", ";").replace(";", "")    # Remove semicolons
    sql = re.sub(r'\s*=\s*', '=', sql)               # Remove spaces around equals
    sql = re.sub(r'\s*,\s*', ',', sql)               # Remove spaces around commas
    sql = sql.strip()
    return sql

def is_exact_match(generated, expected):
    return normalize_sql(generated) == normalize_sql(expected)


def run_query(conn, query):
    try:
        return pd.read_sql_query(query, conn)
    except Exception as e:
        print(f"Query failed: {e}")
        return None

def normalize_result(df):
    if df is None or df.empty:
        return pd.DataFrame()

    df = df.copy()
    df.columns = [col.lower().strip() for col in df.columns]
    df = df.map(lambda x: str(x).strip().lower())
    df = df.sort_index(axis=1).sort_values(by=df.columns.tolist(), ignore_index=True)
    return df.reset_index(drop=True)

def is_execution_correct(generated_sql, expected_sql, conn):
    try:
        gen_result = run_query(conn, generated_sql)
        exp_result = run_query(conn, expected_sql)
    except Exception as e:
        print(f"Query error: {e}")
        return False

    if gen_result is None or exp_result is None:
        return False

    try:
        norm_gen = normalize_result(gen_result)
        norm_exp = normalize_result(exp_result)
        return norm_gen.equals(norm_exp)
    except Exception as e:
        print(f"Comparison error: {e}")
        return False



In [17]:
import csv

exact_matches = 0
exec_matches = 0
results = []

# Evaluation loop
for i, item in enumerate(examples):
    question = item["question"]
    expected = item["expected_sql"]
    db_id = item["db_id"]
    db_path = f"spider/spider/database/{db_id}/{db_id}.sqlite"
    schema = get_schema(db_path)

    generated = generate_sql(question, schema)

    print(f"\n--- Example {i+1} ({db_id}) ---")
    print(f"Q: {question}")
    print(f"Expected SQL: {expected}")
    print(f"Generated SQL: {generated}")

    match = is_exact_match(generated, expected)
    if match:
        print("Exact Match")
        exact_matches += 1
    else:
        print("No Match")

    try:
        conn = sqlite3.connect(db_path)
        exec_match = is_execution_correct(generated, expected, conn)
        conn.close()
    except Exception as e:
        print(f"Execution failed: {e}")
        exec_match = False

    if exec_match:
        print("Execution Match")
        exec_matches += 1
    else:
        print("Execution Mismatch")

    results.append({
        "example_number": i + 1,
        "db_id": db_id,
        "question": question,
        "expected_sql": expected,
        "generated_sql": generated,
        "exact_match": "Match" if match else "Fail",
        "execution_match": "Match" if exec_match else "Fail"
    })

# --- After evaluation, save results to CSV ---
csv_filename = "spider_gpt4o_results.csv"

with open(csv_filename, mode="w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["example_number", "db_id", "question", "expected_sql", "generated_sql", "exact_match", "execution_match"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(results)

print(f"\nResults saved to {csv_filename}")

# --- Print final totals ---
total = len(examples)
print(f"\n Exact Match Accuracy: {exact_matches}/{total} = {exact_matches/total:.2%}")
print(f"Execution Accuracy: {exec_matches}/{total} = {exec_matches/total:.2%}")

# Optionally: Save summary to another small CSV
summary_filename = "spider_gpt4o_summary.csv"
with open(summary_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerow(["exact_match_accuracy", f"{exact_matches}/{total} ({exact_matches/total:.2%})"])
    writer.writerow(["execution_accuracy", f"{exec_matches}/{total} ({exec_matches/total:.2%})"])

print(f" Summary saved to {summary_filename}")



--- Example 1 (concert_singer) ---
Q: How many singers do we have?
Expected SQL: SELECT count(*) FROM singer
Generated SQL: SELECT count(*) FROM singer
Exact Match
Execution Match

--- Example 2 (concert_singer) ---
Q: What is the total number of singers?
Expected SQL: SELECT count(*) FROM singer
Generated SQL: SELECT count(*) FROM singer
Exact Match
Execution Match

--- Example 3 (concert_singer) ---
Q: Show name, country, age for all singers ordered by age from the oldest to the youngest.
Expected SQL: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Generated SQL: SELECT name, country, age FROM singer ORDER BY age DESC
Exact Match
Execution Match

--- Example 4 (concert_singer) ---
Q: What are the names, countries, and ages for every singer in descending order of age?
Expected SQL: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Generated SQL: SELECT name, country, age FROM singer ORDER BY age DESC
Exact Match
Execution Match

--- Example 5 (concert_singer) -

In [15]:
print(f"\n Exact Match Accuracy: {exact_matches}/{len(examples)} = {exact_matches/len(examples):.2%}")
print(f" Execution Accuracy: {exec_matches}/{len(examples)} = {exec_matches/len(examples):.2%}")



🧠 Exact Match Accuracy: 379/1034 = 36.65%
🧪 Execution Accuracy: 701/1034 = 67.79%


In [11]:
import json

finetune_data = []

for item in examples:
    schema = get_schema(f"spider/spider/database/{item['db_id']}/{item['db_id']}.sqlite")
    finetune_example = {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant that translates natural language questions into SQL queries given a schema."},
            {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion:\n{item['question']}"},
            {"role": "assistant", "content": item["expected_sql"]}
        ]
    }
    finetune_data.append(finetune_example)

fine_tune_filename = "spider_finetune_data.jsonl"

with open(fine_tune_filename, "w", encoding="utf-8") as f:
    for example in finetune_data:
        f.write(json.dumps(example) + "\n")

print(f"Fine-tuning file saved as {fine_tune_filename}")


✅ Fine-tuning file saved as spider_finetune_data.jsonl
